# ConstrucciÃƒÆ’Ã‚Â³n del panel analÃƒÆ’Ã‚Â­tico integrado 2024

1. **Objetivo del notebook**

- Integrar ambiente, entomologÃƒÆ’Ã‚Â­a y epidemiologÃƒÆ’Ã‚Â­a en un panel de grilla-semana.

2. **Decisiones metodolÃƒÆ’Ã‚Â³gicas**

- Usar el panel ambiental como universo de observaciones.
- Realizar uniones uno a uno y conservar los valores ausentes sin imputar ceros.

3. **Fuentes**

- `data/processed/environment/environment_grid_week_2024_clean.parquet`.
- `data/processed/entomology/entomology_2024_clean.parquet`.
- `data/processed/epidemiology/epidemiology_2024_grid_week.parquet`.

4. **Productos**

- `data/processed/integrated/analytical_grid_week_2024.parquet`.


Cada fila representa una combinaciÃƒÆ’Ã‚Â³n ÃƒÆ’Ã‚Âºnica de `grid_id`, `year` y `epiweek`. Este notebook sÃƒÆ’Ã‚Â³lo integra productos ya preparados.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import HTML, display

for candidate in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "paths.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("No se encontrÃƒÆ’Ã‚Â³ src/paths.py.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import (
    ANALYTICAL_GRID_WEEK_PATH,
    ENVIRONMENT_GRID_WEEK_CLEAN_PATH,
    ENTOMOLOGY_CLEAN_PATH,
    EPIDEMIOLOGY_GRID_WEEK_PATH,
)

KEY = ["grid_id", "year", "epiweek"]

## 1. Entradas y normalizaciÃƒÆ’Ã‚Â³n de llaves

Se cargan los tres paneles canÃƒÆ’Ã‚Â³nicos y se homologan los tipos de `grid_id`, `year` y `epiweek`.

In [2]:
input_paths = {
    "environment": ENVIRONMENT_GRID_WEEK_CLEAN_PATH,
    "entomology": ENTOMOLOGY_CLEAN_PATH,
    "epidemiology": EPIDEMIOLOGY_GRID_WEEK_PATH,
}
missing_inputs = {name: str(path) for name, path in input_paths.items() if not path.is_file()}
if missing_inputs:
    raise FileNotFoundError(f"Productos dependientes ausentes: {missing_inputs}")

environment = pd.read_parquet(ENVIRONMENT_GRID_WEEK_CLEAN_PATH)
entomology = pd.read_parquet(ENTOMOLOGY_CLEAN_PATH)
epidemiology = pd.read_parquet(EPIDEMIOLOGY_GRID_WEEK_PATH)

# Normaliza los tipos de las llaves para poder unir los tres productos canÃƒÆ’Ã‚Â³nicos
for frame in (environment, entomology, epidemiology):
    frame["grid_id"] = pd.to_numeric(frame["grid_id"], errors="raise").astype("Int64").astype("string")
    frame["year"] = pd.to_numeric(frame["year"], errors="raise").astype("int64")
    frame["epiweek"] = pd.to_numeric(frame["epiweek"], errors="raise").astype("int64")
display(pd.DataFrame({
    "fuente": ["ambiente", "entomologÃƒÆ’Ã‚Â­a", "epidemiologÃƒÆ’Ã‚Â­a"],
    "filas": [len(environment), len(entomology), len(epidemiology)],
    "semanas": [environment["epiweek"].nunique(), entomology["epiweek"].nunique(), epidemiology["epiweek"].nunique()],
}))

,fuente,filas,semanas
0,ambiente,177268,52
1,entomologÃƒÆ’Ã‚Â­a,19,15
2,epidemiologÃƒÆ’Ã‚Â­a,4,4


## 2. Panel ambiental base

El panel ambiental limpio define el universo de combinaciones `grid_id`ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“semana.

In [3]:
environment = environment.sort_values(KEY).reset_index(drop=True)
print(f"grid_id en el universo ambiental: {environment['grid_id'].nunique():,}")


grid_id en el universo ambiental: 3,409


## 3. EntomologÃƒÆ’Ã‚Â­a

Se integra entomologÃƒÆ’Ã‚Â­a mediante una uniÃƒÆ’Ã‚Â³n izquierda uno a uno. Las combinaciones sin hallazgos conservan valores nulos.

Se incorporan los totales de registros, vectores y etapas del producto limpio. Estos totales reÃƒÆ’Ã‚Âºnen asignaciones exactas y aproximadas; su desglose se conserva en el antecedente entomolÃƒÆ’Ã‚Â³gico y no se calcula en este panel.

In [4]:
env_keys = set(map(tuple, environment[KEY].to_numpy()))
ento_keys = set(map(tuple, entomology[KEY].to_numpy()))
outside_ento = sorted(ento_keys - env_keys)
if outside_ento:
    raise ValueError(f"Llaves entomolÃƒÆ’Ã‚Â³gicas fuera del ambiente ({len(outside_ento)}); ejemplos: {outside_ento[:10]}")
ento_overlap = sorted(set(entomology.columns).intersection(environment.columns) - set(KEY))
ento_rename = {c: f"ento_{c}" for c in ento_overlap}
ento_join = entomology.rename(columns=ento_rename)
integrated = environment.merge(ento_join, on=KEY, how="left", validate="one_to_one", indicator="_ento_merge")
ento_matched = integrated["_ento_merge"].eq("both")
integrated = integrated.drop(columns="_ento_merge")
missing_ento_rows = ~ento_matched
ento_output_columns = [c for c in ento_join.columns if c not in KEY]
if ento_output_columns and integrated.loc[missing_ento_rows, ento_output_columns].notna().any().any():
    raise ValueError("Se creÃƒÆ’Ã‚Â³ informaciÃƒÆ’Ã‚Â³n entomolÃƒÆ’Ã‚Â³gica en una combinaciÃƒÆ’Ã‚Â³n sin hallazgo; revisar uniÃƒÆ’Ã‚Â³n.")
display(pd.DataFrame({"renombre_entomologia": list(ento_rename.items())}))


,renombre_entomologia


## 4. EpidemiologÃƒÆ’Ã‚Â­a

Se integra epidemiologÃƒÆ’Ã‚Â­a mediante una uniÃƒÆ’Ã‚Â³n izquierda uno a uno. Las combinaciones sin registros epidemiolÃƒÆ’Ã‚Â³gicos conservan valores nulos.

In [5]:
epi_keys = set(map(tuple, epidemiology[KEY].to_numpy()))
outside_epi = sorted(epi_keys - env_keys)
if outside_epi:
    raise ValueError(f"Llaves epidemiolÃƒÆ’Ã‚Â³gicas fuera del ambiente ({len(outside_epi)}); ejemplos: {outside_epi[:10]}")
epi_overlap = sorted(set(epidemiology.columns).intersection(integrated.columns) - set(KEY))
epi_rename = {c: f"epi_{c}" for c in epi_overlap}
epi_join = epidemiology.rename(columns=epi_rename)
integrated = integrated.merge(epi_join, on=KEY, how="left", validate="one_to_one", indicator="_epi_merge")
epi_matched = integrated["_epi_merge"].eq("both")
integrated = integrated.drop(columns="_epi_merge")
display(pd.DataFrame({"renombre_epidemiologia": list(epi_rename.items())}))


,renombre_epidemiologia


## 5. AuditorÃƒÆ’Ã‚Â­a, diccionario y guardado

Se revisan las llaves, el esquema final y el diccionario antes de guardar el panel analÃƒÆ’Ã‚Â­tico.

In [6]:
if set(map(tuple, integrated[KEY].to_numpy())) != env_keys:
    raise ValueError("La uniÃƒÆ’Ã‚Â³n perdiÃƒÆ’Ã‚Â³ o creÃƒÆ’Ã‚Â³ llaves respecto del universo ambiental.")
if "geometry" in integrated.columns:
    raise ValueError("El panel integrado no debe contener geometry.")
forbidden_tokens = ("distrit", "comuna", "provincia", "ponder", "weight", "zona_censal")
forbidden_columns = [c for c in integrated.columns if any(t in str(c).lower() for t in forbidden_tokens)]
if forbidden_columns:
    raise ValueError(f"Columnas territoriales o de ponderaciÃƒÆ’Ã‚Â³n no permitidas: {forbidden_columns}")
panel_diagnostics = pd.DataFrame({
    "indicador": [
        "filas_universo", "grilla_semanas_con_hallazgos",
        "grilla_semanas_con_epidemiologia", "porcentaje_con_hallazgos", "porcentaje_con_epidemiologia",
    ],
    "valor": [
        len(integrated),
        int(ento_matched.sum()),
        int(epi_matched.sum()),
        100 * ento_matched.mean(),
        100 * epi_matched.mean(),
    ],
})
display(panel_diagnostics)
display(pd.DataFrame({
    "epiweek": integrated["epiweek"],
    "hallazgos": ento_matched,
    "epidemiologia": epi_matched,
}).groupby("epiweek").agg(
    celdas=("epiweek", "size"),
    hallazgos=("hallazgos", "sum"),
    epidemiologia=("epidemiologia", "sum"),
).reset_index())

dictionary = pd.DataFrame([
    ("grid_id", "Ambiente", "clave", "identificador de celda", "no imputar", "sin faltantes"),
    ("year", "Calendario comÃƒÆ’Ã‚Âºn", "entero", "aÃƒÆ’Ã‚Â±o epidemiolÃƒÆ’Ã‚Â³gico", "no imputar", "2024"),
    ("epiweek", "Calendario comÃƒÆ’Ã‚Âºn", "entero", "semana domingo-sÃƒÆ’Ã‚Â¡bado", "no imputar", "1..52"),
    ("n_cases_confirmed", "EpidemiologÃƒÆ’Ã‚Â­a", "conteo", "casos confirmados asignados", "no rellenar", "combinaciones observadas"),
    ("n_reporting_centers", "EpidemiologÃƒÆ’Ã‚Â­a", "conteo", "centros notificadores ÃƒÆ’Ã‚Âºnicos", "no rellenar", "combinaciones observadas"),
    ("total registros", "Entomolog\u00eda", "conteo", "registros de hallazgos asignados", "conservar nulos", "incluye asignaciones exactas y proxy"),
    ("hallazgos vectores", "Entomolog\u00eda", "conteo", "suma disponible de huevos, larvas, pupas y adultos", "conservar nulos", "incluye asignaciones exactas y proxy"),
    ("huevos", "Entomolog\u00eda", "conteo", "huevos contabilizados", "conservar nulos", "incluye asignaciones exactas y proxy"),
    ("larvas", "Entomolog\u00eda", "conteo", "larvas contabilizadas", "conservar nulos", "incluye asignaciones exactas y proxy"),
    ("pupas", "Entomolog\u00eda", "conteo", "pupas contabilizadas", "conservar nulos", "incluye asignaciones exactas y proxy"),
    ("adultos", "Entomolog\u00eda", "conteo", "adultos contabilizados", "conservar nulos", "incluye asignaciones exactas y proxy"),
], columns=["variable", "fuente", "tipo", "significado", "tratamiento_del_faltante", "condicion_imputacion_o_proxy"])
display(dictionary)

column_renames = {
    "alt_min": "elevation_min",
    "alt_mean": "elevation_mean",
    "alt_max": "elevation_max",
    "koppen_fin": "koppen_class",
    "ndvi_media_bisemanal": "ndvi_mean_biweekly",
    "ndvi_maximo_bisemanal": "ndvi_max_biweekly",
    "ndvi_minimo_bisemanal": "ndvi_min_biweekly",
    "ndvi_desvest_bisemanal": "ndvi_std_biweekly",
    "total registros": "total_records",
    "hallazgos vectores": "vector_findings",
    "huevos": "eggs",
    "larvas": "larvae",
    "pupas": "pupae",
    "adultos": "adults",
    "n_cases_confirmed": "confirmed_cases",
    "n_reporting_centers": "reporting_centers",
}
integrated = integrated.rename(columns=column_renames)
integrated = integrated.sort_values(KEY).reset_index(drop=True)
ANALYTICAL_GRID_WEEK_PATH.parent.mkdir(parents=True, exist_ok=True)
integrated.to_parquet(ANALYTICAL_GRID_WEEK_PATH, index=False)
print(f"Panel analÃƒÆ’Ã‚Â­tico guardado: {ANALYTICAL_GRID_WEEK_PATH}; filas={len(integrated):,}")

,indicador,valor
0,filas_universo,177268.000000
1,grilla_semanas_con_hallazgos,19.000000
2,grilla_semanas_con_epidemiologia,4.000000
3,porcentaje_con_hallazgos,0.010718
4,porcentaje_con_epidemiologia,0.002256


,epiweek,celdas,hallazgos,epidemiologia
0,1,3409,0,0
1,2,3409,0,0
2,3,3409,0,0
3,4,3409,0,0
4,5,3409,1,0
5,6,3409,0,0
6,7,3409,0,1
7,8,3409,0,0
8,9,3409,1,0
9,10,3409,0,0


,variable,fuente,tipo,significado,tratamiento_del_faltante,condicion_imputacion_o_proxy
0,grid_id,Ambiente,clave,identificador de celda,no imputar,sin faltantes
1,year,Calendario comÃƒÆ’Ã‚Âºn,entero,aÃƒÆ’Ã‚Â±o epidemiolÃƒÆ’Ã‚Â³gico,no imputar,2024
2,epiweek,Calendario comÃƒÆ’Ã‚Âºn,entero,semana domingo-sÃƒÆ’Ã‚Â¡bado,no imputar,1..52
3,n_cases_confirmed,EpidemiologÃƒÆ’Ã‚Â­a,conteo,casos confirmados asignados,no rellenar,combinaciones observadas
4,n_reporting_centers,EpidemiologÃƒÆ’Ã‚Â­a,conteo,centros notificadores ÃƒÆ’Ã‚Âºnicos,no rellenar,combinaciones observadas
5,total registros,Entomología,conteo,registros de hallazgos asignados,conservar nulos,incluye asignaciones exactas y proxy
6,hallazgos vectores,Entomología,conteo,"suma disponible de huevos, larvas, pupas y adu...",conservar nulos,incluye asignaciones exactas y proxy
7,huevos,Entomología,conteo,huevos contabilizados,conservar nulos,incluye asignaciones exactas y proxy
8,larvas,Entomología,conteo,larvas contabilizadas,conservar nulos,incluye asignaciones exactas y proxy
9,pupas,Entomología,conteo,pupas contabilizadas,conservar nulos,incluye asignaciones exactas y proxy


Panel analÃƒÆ’Ã‚Â­tico guardado: C:\tesis_aedes_grilla_2024\data\processed\integrated\analytical_grid_week_2024.parquet; filas=177,268


## 6. Vista del parquet integrado

Primeras 15 filas de `analytical_grid_week_2024.parquet`, leídas desde el archivo guardado. La barra inferior permite recorrer horizontalmente todas las columnas.

In [7]:
preview = pd.read_parquet(ANALYTICAL_GRID_WEEK_PATH).head(15)
display(HTML(
    '<div style="width:100%; overflow-x:auto; padding-bottom:8px;">'
    + preview.to_html(index=False, max_cols=None)
    + '</div>'
))


grid_id,year,epiweek,tmean_c,tmin_c,tmax_c,elevation_min,elevation_mean,elevation_max,rh_mean,rh_min,rh_max,tp_sum,koppen_class,population,ndvi_mean_biweekly,ndvi_max_biweekly,ndvi_min_biweekly,ndvi_std_biweekly,epiweek_sin,epiweek_cos,total_records,vector_findings,eggs,larvae,pupae,adults,confirmed_cases,reporting_centers
0,2024,1,21.209301,19.992899,22.276896,25.0,33.596774,43.0,74.356583,71.468781,76.885185,0.016606,BWh,0.0,0.050019,0.081831,-0.021624,0.004055,0.120537,9.927089e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,2,22.054096,21.573660,22.765982,25.0,33.596774,43.0,73.982315,71.317276,76.284805,0.011030,BWh,0.0,0.048979,0.080882,-0.004966,0.003381,0.239316,9.709418e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,3,22.568083,22.038534,23.052988,25.0,33.596774,43.0,75.665398,72.660240,78.163437,0.068585,BWh,0.0,0.047940,0.079934,0.011691,0.002707,0.354605,9.350162e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,4,23.774378,23.398520,24.111084,25.0,33.596774,43.0,72.535446,70.525070,74.601913,0.008243,BWh,0.0,0.043426,0.070744,0.004822,0.001353,0.464723,8.854560e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,5,22.861515,22.499634,23.158884,25.0,33.596774,43.0,73.460915,72.232567,74.605240,0.005395,BWh,0.0,0.038912,0.061555,-0.002047,0.000000,0.568065,8.229839e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,6,25.018829,24.110687,25.648270,25.0,33.596774,43.0,71.713272,67.142258,77.242050,0.005812,BWh,0.0,0.028242,0.242594,-0.162309,0.000676,0.663123,7.485107e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,7,23.730053,23.066925,24.236368,25.0,33.596774,43.0,75.282051,71.676620,81.074158,0.052671,BWh,0.0,0.033400,0.316837,-0.140295,0.000000,0.748511,6.631227e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,8,24.646017,23.532104,25.408484,25.0,33.596774,43.0,76.529297,72.132515,81.423096,0.047258,BWh,0.0,0.032289,0.323296,-0.150157,0.000000,0.822984,5.680647e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,9,23.541357,22.868347,24.530279,25.0,33.596774,43.0,75.064690,73.228439,77.841995,0.171275,BWh,0.0,0.027595,0.062425,-0.020239,0.000000,0.885456,4.647232e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
0,2024,10,23.356249,22.439676,24.117676,25.0,33.596774,43.0,74.060120,72.234764,77.018234,0.004876,BWh,0.0,0.037177,0.069368,-0.017917,0.003663,0.935016,3.546049e-01,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
